# 从手里的一笔钱开始

我们先不急着谈股票、债券、期权、期货。我们只问一个最朴素的问题：

> 如果我现在有一笔钱，未来某个时间点它会变成什么？

这句话里面其实已经藏着金融工程最底层的几个原语：

1. 现在
2. 未来
3. 时间间隔
4. 金额
5. 现金流
6. 利息
7. 利率
8. 贴现
9. 不确定性

我们会从“我”和“Tom”之间的一笔借款开始，一步一步把这些原语手搓出来。

## 1. 如果一笔现金放着不动，会发生什么？

假设我现在手里有 1000 元。

如果我把它放在抽屉里，不借给别人，不存银行，也不投资。

那么过了一段时间之后，它在名义金额上仍然是 1000 元。

这里有一个很重要的区分：

- 名义金额：纸面上写着多少元。
- 购买力：这笔钱未来能买多少东西。

在金融工程最开始，我们先只讨论名义金额。

如果不发生任何交易、不产生任何利息，那么：

$$A = P$$

其中：

- $P$ 表示现在的本金（Principle）。
- $A$ 表示未来拿到的金额（Amount）。

这个公式很简单，但它是一个基准世界：

> 钱如果没有被使用，它不会自己在名义上变多。

In [1]:
def future_amount_if_idle(principal):
    """
    一笔钱如果只是放着不动，名义金额不变。
    """
    return principal


P = 1000
A = future_amount_if_idle(P)

print("现在的本金:", P)
print("未来的金额:", A)

现在的本金: 1000
未来的金额: 1000


目前我们只得到一个最朴素的函数：

```python
future_amount_if_idle(principal)
```

它几乎什么金融知识都没有。但如果未来金额变多了，一定是因为中间发生了什么。

## 2. Tom 向我借钱：为什么他要付出代价？

现在 Tom 找到我，说：

> 我想借你 1000 元，过一段时间还你。

如果 Tom 未来只还我 1000 元，我可能会犹豫。

因为我把钱借给 Tom 以后，会失去几件东西：

1. 我暂时不能使用这笔钱。
2. Tom 未来可能还不上。
3. 我可能错过其他投资机会。
4. 未来的钱可能没有现在的钱有用。

所以 Tom 通常需要多还一点。

假设 Tom 现在借走 1000 元，未来还我 1050 元。

多出来的 50 元，就是我的额外收益，或者说好处，金融上叫利息。

$$I = A - P$$

其中：

- $I$ 是利息（Interest）。
- $P$ 是本金。
- $A$ 是未来还款金额。

In [ ]:
def interest_amount(principal, repayment):
    """
    计算利息。
    principal: 现在借出去的钱
    repayment: 未来收回的钱
    """
    return repayment - principal


P = 1000
A = 1050
I = interest_amount(P, A)

print("本金:", P)
print("未来还款:", A)
print("利息:", I)

## 3. 衡量收益、好处的东西

我们只知道：

- Tom 借了 1000 元。
- 未来某个时间点还了 1050 元。
- 利息是 50 元。

那么这段时间内，我赚了多少比例？

这个比例叫做这段时间的利率，或者更准确地说，叫做这段时间的收益率。

$$r_{T} = \frac{A - P}{P}$$

这里的 $T$ 提醒我们：

> 这个利率是某个时间长度上的利率。

In [10]:
def period_rate(principal, repayment):
    """
    计算某一段时间内的利率。
    注意：这不是年化利率。
    """
    return (repayment - principal) / principal


P = 1000
A = 1050

r_T = period_rate(P, A)

print("这段时间内的利率:", r_T)
print("百分比形式:", r_T * 100, "%")

这段时间内的利率: 0.05
百分比形式: 5.0 %


### 解释一下这个结果

输出结果是 0.05，也就是 5%。

这意味着：

> Tom 在这段时间内，为了使用我的 1000 元，额外支付了 5% 的代价。

但我们还没有说这段时间是一个月、半年，还是一年。

请注意：

> 利率一定要和时间长度绑定在一起。

## 4. 引入时间点：现在和未来

金融工程里面的很多错误，都来自于没有说清楚“钱发生在什么时候”。

所以我们从现在开始引入时间点。

我们用 $t = 0$ 表示现在。

用 $t = 1$ 表示未来某个标准时间单位之后。

这个标准时间单位可以是一年，也可以是一个月。

在本讲里，我们先把 $t$ 当成年数。

所以：

- $t = 0$ 表示现在。
- $t = 0.5$ 表示半年后。
- $t = 1$ 表示一年后。
- $t = 2$ 表示两年后。

我们先手搓一个时间线生成器。

In [3]:
def make_time_grid(start, end, step):
    """
    生成从 start 到 end 的时间点。
    这里用浮点数表示时间，单位暂时是“年”。
    """
    points = []
    t = start

    # 为了避免浮点误差，用一个小容忍度
    eps = 0.000000001

    while t <= end + eps:
        points.append(round(t, 10))
        t = t + step

    return points


time_grid = make_time_grid(0, 2, 0.5)

print(time_grid)

[0, 0.5, 1.0, 1.5, 2.0]


现在我们已经可以表示一个简单时间轴：

```text
0      0.5      1.0      1.5      2.0
|-------|--------|--------|--------|
现在    半年后    一年后    一年半后   两年后
```

这个时间轴以后会被反复使用。

任何金融产品都必须回答一个问题：

> 它在哪些时间点产生现金流？

In [6]:
def draw_time_grid(points):
    """
    用 ASCII 画一个最小时间线。
    """
    line = ""
    labels = ""

    for p in points:
        mark = "|"
        text = str(p)

        line += mark + "------"
        labels += text + "    "

    print(line)
    print(labels)


draw_time_grid(time_grid)

|------|------|------|------|------
0    0.5    1.0    1.5    2.0    


## 5. 现金流：钱在某个时间点流入或流出

现在我们把 Tom 的借款放到时间线上。

从我的角度看：

- 在 $t = 0$，我借给 Tom 1000 元，所以我现金流出 1000 元。
- 在 $t = 1$，Tom 还给我 1050 元，所以我现金流入 1050 元。

从Tom的角度看：

- 在 $t = 0$，我借给 Tom 1000 元，所以Tom现金流入 1000 元。
- 在 $t = 1$，Tom 还给我 1050 元，所以Tom现金流出 1050 元。

我们约定：

- 现金流入记为正数。
- 现金流出记为负数。

所以这笔交易对于我来说可以表示为：

```text
t = 0:  -1000
t = 1:  +1050
```

这就是现金流。金融工程里非常多的东西，本质上都是现金流。

In [7]:
class CashFlow:
    """
    一笔现金流。
    time: 发生时间
    amount: 金额，正数表示流入，负数表示流出
    label: 说明
    """
    def __init__(self, time, amount, label=""):
        self.time = time
        self.amount = amount
        self.label = label

    def __repr__(self):
        return f"CashFlow(time={self.time}, amount={self.amount}, label='{self.label}')"


cf1 = CashFlow(0, -1000, "lend money to Tom")
cf2 = CashFlow(1, 1050, "Tom repays money")

print(cf1)
print(cf2)

CashFlow(time=0, amount=-1000, label='lend money to Tom')
CashFlow(time=1, amount=1050, label='Tom repays money')


## 6. 现金流表：多笔现金流组成一笔交易

一笔交易通常不是一个现金流，而是一组现金流。

比如借钱给 Tom：

- 现在流出 1000。
- 一年后流入 1050。

我们把它封装成现金流表。

In [8]:
class CashFlowSchedule:
    """
    一组现金流。
    """
    def __init__(self, cashflows):
        self.cashflows = cashflows

    def total_amount(self):
        """
        直接把所有现金流金额相加。
        注意：这不是现值，只是名义金额求和。
        """
        total = 0
        for cf in self.cashflows:
            total = total + cf.amount
        return total

    def print_table(self):
        print("time\tamount\tlabel")
        for cf in self.cashflows:
            print(str(cf.time) + "\t" + str(cf.amount) + "\t" + cf.label)

    def __repr__(self):
        return repr(self.cashflows)


tom_loan = CashFlowSchedule([
    CashFlow(0, -1000, "I lend money to Tom"),
    CashFlow(1, 1050, "Tom repays")
])

tom_loan.print_table()
print("名义金额直接相加:", tom_loan.total_amount())

time	amount	label
0	-1000	I lend money to Tom
1	1050	Tom repays
名义金额直接相加: 50


### 请注意：不要把现金流直接相加当成价值

上面直接相加得到 50。

但这只是：

```text
未来收到 1050，今天付出 1000，名义差额是 50。
```

它还不是严格意义上的“现在值多少钱”。

因为 1050 发生在未来，1000 发生在现在。

我们用几笔交易巩固“时间 + 现金流 + 利率”。

## 7. 三笔 Tom 交易：引入时间长度

现在 Tom 给我三个选择：

交易 A：

- 现在借走 1000。
- 半年后还 1030。

交易 B：

- 现在借走 1000。
- 一年后还 1060。

交易 C：

- 现在借走 1000。
- 两年后还 1120。

如果只看多还了多少钱：

- A 多还 30。
- B 多还 60。
- C 多还 120。

看起来 C 最多。

但 C 占用了我的钱两年。

所以我们需要把时间也放进去比较。

In [11]:
deals = [
    {"name": "A", "principal": 1000, "repayment": 1030, "T": 0.5},
    {"name": "B", "principal": 1000, "repayment": 1060, "T": 1.0},
    {"name": "C", "principal": 1000, "repayment": 1120, "T": 2.0},
]

for deal in deals:
    r_T = period_rate(deal["principal"], deal["repayment"])
    print("交易", deal["name"])
    print("  时间长度:", deal["T"], "年")
    print("  期间利率:", r_T)
    print("  百分比:", r_T * 100, "%")

交易 A
  时间长度: 0.5 年
  期间利率: 0.03
  百分比: 3.0 %
交易 B
  时间长度: 1.0 年
  期间利率: 0.06
  百分比: 6.0 %
交易 C
  时间长度: 2.0 年
  期间利率: 0.12
  百分比: 12.0 %


## 8. 年化利率--一个统一的标准

为什么刚才不一开始就讲年化利率？

因为年化利率是为了比较不同时间长度的交易。

如果三笔交易的时间不同，我们需要把它们换算成“每一年大概是多少”。

第一种换算方式叫简单年化。

它假设收益是按时间线性增长的。

$$r_{\text{annual simple}} = \frac{\frac{A}{P} - 1}{T}$$

其中：

- $A$ 是未来还款。
- $P$ 是现在本金。
- $T$ 是以年为单位的时间长度。

注意：

> 简单年化只是一个比较工具，不一定代表真实的复利过程。

In [12]:
def simple_annual_rate(principal, repayment, T):
    """
    简单年化利率。
    """
    return ((repayment / principal) - 1) / T


for deal in deals:
    r_annual_simple = simple_annual_rate(
        deal["principal"],
        deal["repayment"],
        deal["T"]
    )
    print("交易", deal["name"], "简单年化利率:", r_annual_simple, "百分比:", r_annual_simple * 100, "%")

交易 A 简单年化利率: 0.06000000000000005 百分比: 6.000000000000005 %
交易 B 简单年化利率: 0.06000000000000005 百分比: 6.000000000000005 %
交易 C 简单年化利率: 0.06000000000000005 百分比: 6.000000000000005 %


这样看来，三笔交易的收益率是不是一样的【这里需要补充文本，说明不能只考虑利率，要考虑期限和另一件事；然后再过渡，时间续上的4A也不等于2B也不等于C，然后引出了复利】

## 9. 复利年化：如果收益会再产生收益

简单年化有一个隐含假设：

> 半年赚 3%，一年就赚 6%。

但现实中，如果半年后赚到的钱也可以继续投资，那么收益会再产生收益。

这就是复利思想。

如果一笔钱每年按照同一个年利率 $r$ 增长，经过 $T$ 年后：

$$A = P(1 + r)^{T}$$

现在反过来，已知 $P$、$A$ 和 $T$，可以求复利年化利率：

$$r_{\text{annual compound}} = \left(\frac{A}{P}\right)^{\frac{1}{T}} - 1$$

In [ ]:
def compound_annual_rate(principal, repayment, T):
    """
    复利年化利率。
    """
    return (repayment / principal) ** (1 / T) - 1


for deal in deals:
    r_annual_compound = compound_annual_rate(
        deal["principal"],
        deal["repayment"],
        deal["T"]
    )
    print("交易", deal["name"], "复利年化利率:", r_annual_compound, "百分比:", r_annual_compound * 100, "%")

### 小结：到这里我们有了什么？

我们已经有了四个原语：

1. 时间点
2. 现金流
3. 利息
4. 利率

以及两个利率比较工具：

1. 简单年化利率
2. 复利年化利率

但我们还有一个问题没有解决：

> 未来收到的钱，今天到底值多少钱？

这就是贴现。

## 10. 贴现：把未来的钱搬回现在

假设市场上有一个相对安全的机会：

> 今天投入 1000 元，一年后可以得到 1100 元。

也就是说，一年收益率是 10%。

那么如果 Tom 承诺一年后给我 1100 元，这个承诺今天值多少钱？

答案是 1000 元。

因为我今天拿 1000 元去做这个安全机会，一年后也能得到 1100 元。

所以未来的 1100 元，在年利率 10% 的世界里，今天的价值是 1000 元。

这个“把未来金额换算成现在金额”的动作，叫贴现。

如果一年期利率是 $r$，未来金额是 $FV$，那么现值是：

$$PV = \frac{FV}{1 + r}$$

如果是 $T$ 年复利贴现：

$$PV = \frac{FV}{(1 + r)^{T}}$$

In [ ]:
def discount_factor(rate, T):
    """
    复利贴现因子。
    rate: 年利率
    T: 年数
    """
    return 1 / ((1 + rate) ** T)


def present_value(future_value, rate, T):
    """
    把未来金额贴现到现在。
    """
    return future_value * discount_factor(rate, T)


FV = 1100
r = 0.10
T = 1

PV = present_value(FV, r, T)

print("未来金额:", FV)
print("利率:", r)
print("时间:", T)
print("现值:", PV)

### 贴现的直觉

贴现不是在说未来的钱“变少了”。

它是在问：

> 如果我想在未来得到这笔钱，今天需要准备多少钱？

所以贴现是未来价值和当前价值之间的桥。

未来金额越远，现值通常越低。

利率越高，现值通常越低。

In [ ]:
future_value = 1000
rates = [0.01, 0.05, 0.10]
times = [1, 2, 5]

for rate in rates:
    for T in times:
        pv = present_value(future_value, rate, T)
        print("未来", T, "年后收到", future_value, "利率", rate, "现值", pv)

## 11. 净现值：把一组现金流都搬回现在

现在我们回到现金流表。

一组现金流可能发生在不同时间。

要比较它们，最自然的方法是：

> 把每一笔未来现金流都贴现到现在，然后相加。

这叫净现值。

对于现金流 $CF_{k}$，发生在时间 $t_{k}$，如果贴现率是 $r$，则：

$$PV_{k} = \frac{CF_{k}}{(1 + r)^{t_{k}}}$$

所有现金流的净现值是：

$$NPV = \sum_{k=1}^{n} \frac{CF_{k}}{(1 + r)^{t_{k}}}$$

In [ ]:
def npv(cashflow_schedule, rate):
    """
    计算一组现金流的净现值。
    """
    total = 0

    for cf in cashflow_schedule.cashflows:
        pv = present_value(cf.amount, rate, cf.time)
        total = total + pv

    return total


tom_loan = CashFlowSchedule([
    CashFlow(0, -1000, "I lend money to Tom"),
    CashFlow(1, 1050, "Tom repays")
])

for rate in [0.00, 0.03, 0.05, 0.10]:
    value = npv(tom_loan, rate)
    print("贴现率:", rate, "Tom 这笔交易对我的净现值:", value)

### 解释输出

如果贴现率是 0%，那么未来的 1050 不打折，净现值就是 50。

如果贴现率是 5%，那么一年后的 1050 折回今天刚好是 1000，净现值接近 0。

如果贴现率是 10%，未来的 1050 折回今天不到 1000，净现值是负数。

所以同一笔现金流，在不同贴现率下价值不同。

这句话非常重要：

> 金融产品的价值不是只由现金流决定的，还由贴现率决定。

## 12. 商品与标的物：金融世界里被定价的东西

到目前为止，我们一直在讨论“钱”。

但金融市场不只交易钱，还交易很多东西：

- 黄金
- 石油
- 股票
- 债券
- 外汇
- 玉米
- 比特币

我们先给这些东西一个非常朴素的名字：

> 标的物。

标的物的核心属性是：

> 它在某个时间点有一个价格。

比如：

```text
t = 0，苹果公司股票价格是 100。
t = 1，苹果公司股票价格可能是 110。
```

注意，我们现在不研究股票为什么涨跌。

我们只是引入一个最小原语：

> 某个东西在某个时间点的价格。

In [ ]:
class AssetPrice:
    """
    一个标的物在某个时间点的价格。
    """
    def __init__(self, asset_name, time, price):
        self.asset_name = asset_name
        self.time = time
        self.price = price

    def __repr__(self):
        return f"AssetPrice(asset_name='{self.asset_name}', time={self.time}, price={self.price})"


apple_now = AssetPrice("Apple Stock", 0, 100)

print(apple_now)

这个概念以后会非常重要。

因为期权、期货、远期这些衍生品，都是基于某个标的物设计出来的。

但在原语层，我们先不急着做衍生品。

我们只需要知道：

> 金融合约常常不是直接关于钱，而是关于某个标的物未来的价格。

## 13. 不确定性：Tom 未来一定还钱吗？

到目前为止，我们都假设 Tom 一定会还钱。

但现实中，未来是不确定的。

假设 Tom 借了 1000 元去做生意。

一年后可能出现三种情况：

1. 生意失败，只能还 800。
2. 生意一般，还 1050。
3. 生意很好，还 1300。

这时候未来现金流就不是一个固定数字，而是一个随机变量。

随机变量的意思不是“它乱来”。

它的意思是：

> 现在不知道未来具体是多少，但知道它可能有哪些结果。

## 14. 手搓期望值

如果三种情况的概率分别是：

- 失败：20%
- 一般：60%
- 很好：20%

那么未来还款金额的期望是：

$$E[X] = \sum_{k=1}^{n} p_{k} x_{k}$$

在这个例子里：

$$E[X] = 0.2 \times 800 + 0.6 \times 1050 + 0.2 \times 1300$$

期望值不是保证会发生的值。

它只是长期平均意义上的中心。

In [ ]:
def expected_value(outcomes):
    """
    outcomes 是一个列表。
    每个元素是一个字典：
    {
        "probability": 概率,
        "value": 结果
    }
    """
    total = 0

    for item in outcomes:
        total = total + item["probability"] * item["value"]

    return total


tom_outcomes = [
    {"name": "bad", "probability": 0.2, "value": 800},
    {"name": "normal", "probability": 0.6, "value": 1050},
    {"name": "good", "probability": 0.2, "value": 1300},
]

E = expected_value(tom_outcomes)

print("Tom 未来还款的期望值:", E)
print("相对于本金的期望收益率:", period_rate(1000, E))

### 期望值的陷阱

期望还款是 1050。

这看起来和“确定还款 1050”一样。

但它们不是一回事。

因为确定还款 1050 没有坏情况。

而随机还款中，Tom 有 20% 的概率只能还 800。

所以：

> 期望值相同，不代表风险相同。

后面的金融工程会反复遇到这个问题：

- 价格是一个数。
- 风险是一整个分布。

## 15. 手搓一个最小随机数发生器

为了模拟未来，我们需要随机数。

但本讲不使用 Python 自带的 `random`。

我们手搓一个最小的线性同余随机数发生器。

它的形式是：

$$x_{n+1} = (a x_{n} + c) \bmod m$$

然后把 $x_{n}$ 除以 $m$，得到一个 0 到 1 之间的数。

这不是高质量随机数发生器。

但它足够让我们理解：

> 模拟的核心是用一串看起来随机的数字，生成不同未来。

In [ ]:
class LinearCongruentialGenerator:
    """
    最小线性同余随机数发生器。
    """
    def __init__(self, seed=1):
        self.state = seed
        self.a = 1103515245
        self.c = 12345
        self.m = 2 ** 31

    def next_int(self):
        self.state = (self.a * self.state + self.c) % self.m
        return self.state

    def next_float(self):
        return self.next_int() / self.m


rng = LinearCongruentialGenerator(seed=42)

for _ in range(5):
    print(rng.next_float())

## 16. 用随机数模拟 Tom 的未来

现在我们用刚才的随机数来决定 Tom 一年后的还款。

规则如下：

- 如果随机数小于 0.2，Tom 还 800。
- 如果随机数在 0.2 到 0.8 之间，Tom 还 1050。
- 如果随机数大于等于 0.8，Tom 还 1300。

In [ ]:
def sample_tom_repayment(rng):
    u = rng.next_float()

    if u < 0.2:
        return 800
    elif u < 0.8:
        return 1050
    else:
        return 1300


rng = LinearCongruentialGenerator(seed=7)

for _ in range(10):
    print(sample_tom_repayment(rng))

## 17. 模拟很多个未来

如果我们只模拟一次，结果可能是 800，也可能是 1050，也可能是 1300。

但如果模拟很多次，平均结果应该接近期望值。

这就是蒙特卡洛思想的最小版本：

> 用很多个可能的未来，估计未来结果的平均行为。

In [ ]:
def simulate_many_repayments(seed, count):
    rng = LinearCongruentialGenerator(seed=seed)
    results = []

    for _ in range(count):
        repayment = sample_tom_repayment(rng)
        results.append(repayment)

    return results


def average(values):
    total = 0
    for v in values:
        total = total + v
    return total / len(values)


for count in [10, 100, 1000, 10000]:
    results = simulate_many_repayments(seed=123, count=count)
    avg = average(results)
    print("模拟次数:", count, "平均还款:", avg)

### 到这里我们已经有了随机性的原语

我们已经可以表示：

1. 未来有多种结果。
2. 每种结果有概率。
3. 可以计算期望值。
4. 可以用随机数模拟未来。

这为后面的路径模拟、期权定价和风险管理打基础。

## 18. 连续复利之前：从一年复利到多次复利

现在我们回到利息。

如果年利率是 10%，本金是 1000，一年后按照一年复利：

$$A = 1000 \times (1 + 0.1)$$

得到 1100。

但如果利息半年结算一次呢？

那么每半年利率是 5%，一年结算两次：

$$A = 1000 \times \left(1 + \frac{0.1}{2}\right)^{2}$$

如果每年结算 $n$ 次：

$$A = P \left(1 + \frac{r}{n}\right)^{nT}$$

其中：

- $P$ 是本金。
- $r$ 是年利率。
- $n$ 是每年复利次数。
- $T$ 是年数。

In [ ]:
def compound_future_value(principal, annual_rate, years, compounds_per_year):
    """
    每年复利 compounds_per_year 次的未来值。
    """
    n = compounds_per_year
    return principal * (1 + annual_rate / n) ** (n * years)


P = 1000
r = 0.10
T = 1

for n in [1, 2, 4, 12, 365]:
    fv = compound_future_value(P, r, T, n)
    print("每年复利次数:", n, "未来值:", fv)

## 19. 连续复利：如果复利次数越来越多

当每年复利次数越来越多，未来值会慢慢接近一个极限。

这个极限和数学常数 $e$ 有关。

$e$ 可以这样定义：

$$e = \lim_{n \to \infty} \left(1 + \frac{1}{n}\right)^{n}$$

更一般地：

$$\lim_{n \to \infty} \left(1 + \frac{x}{n}\right)^{n} = e^{x}$$

所以连续复利下：

$$FV_{\text{continuous}} = P e^{rT}$$

贴现时则是：

$$PV_{\text{continuous}} = FV e^{-rT}$$

我们不直接调用 `math.exp`。

我们用级数手搓 $e^{x}$：

$$e^{x} = \sum_{k=0}^{\infty} \frac{x^{k}}{k!}$$

在代码里，我们只取有限项近似。

In [ ]:
def factorial(n):
    """
    手搓阶乘。
    """
    result = 1
    for k in range(1, n + 1):
        result = result * k
    return result


def exp_series(x, terms=30):
    """
    用泰勒级数近似 e 的 x 次方。
    """
    total = 0

    for k in range(terms):
        total = total + (x ** k) / factorial(k)

    return total


def continuous_future_value(principal, annual_rate, years):
    """
    连续复利未来值。
    """
    return principal * exp_series(annual_rate * years)


def continuous_present_value(future_value, annual_rate, years):
    """
    连续复利贴现。
    """
    return future_value * exp_series(-annual_rate * years)


P = 1000
r = 0.10
T = 1

fv_cont = continuous_future_value(P, r, T)

print("连续复利未来值:", fv_cont)

### 比较不同复利频率和连续复利

In [ ]:
P = 1000
r = 0.10
T = 1

print("本金:", P)
print("年利率:", r)
print("年数:", T)
print()

for n in [1, 2, 4, 12, 365, 10000]:
    fv = compound_future_value(P, r, T, n)
    print("每年复利", n, "次:", fv)

print("连续复利:", continuous_future_value(P, r, T))

### 连续复利的直觉

连续复利不是说现实中真的每一瞬间都有人给你结息。

它更像是一种数学上的极限工具。

它有几个好处：

1. 公式更简洁。
2. 多期相加更方便。
3. 后面处理随机价格过程时非常自然。

但在原语层，你只需要记住：

> 连续复利是“复利次数无限增加”时的极限版本。

## 20. 原语层整合：做一个最小金融原语工具箱

现在我们把本讲的组件整理一下。

我们已经拥有：

1. 时间线：`make_time_grid`
2. 单笔现金流：`CashFlow`
3. 现金流表：`CashFlowSchedule`
4. 期间利率：`period_rate`
5. 简单年化：`simple_annual_rate`
6. 复利年化：`compound_annual_rate`
7. 贴现因子：`discount_factor`
8. 现值：`present_value`
9. 净现值：`npv`
10. 标的物价格：`AssetPrice`
11. 随机数发生器：`LinearCongruentialGenerator`
12. 随机还款模拟：`sample_tom_repayment`
13. 期望值：`expected_value`
14. 连续复利：`continuous_future_value`
15. 连续贴现：`continuous_present_value`

这些就是后面组合层的“逻辑门”。

## 21. 最小综合挑战：Tom 的随机借款今天值多少钱？

现在 Tom 要借 1000 元，一年后还款不确定：

- 20% 概率还 800。
- 60% 概率还 1050。
- 20% 概率还 1300。

市场上一年贴现率是 5%。

问：

1. Tom 未来还款的期望是多少？
2. 这个期望还款贴现到今天是多少？
3. 如果我今天借给 Tom 1000 元，这笔交易的期望净现值是多少？

In [ ]:
tom_outcomes = [
    {"name": "bad", "probability": 0.2, "value": 800},
    {"name": "normal", "probability": 0.6, "value": 1050},
    {"name": "good", "probability": 0.2, "value": 1300},
]

expected_repayment = expected_value(tom_outcomes)
discount_rate = 0.05
T = 1

pv_expected_repayment = present_value(expected_repayment, discount_rate, T)

initial_cashflow = -1000
expected_npv = initial_cashflow + pv_expected_repayment

print("期望还款:", expected_repayment)
print("期望还款现值:", pv_expected_repayment)
print("期望净现值:", expected_npv)

### 解释

如果只看期望，Tom 一年后的还款是 1050。

在 5% 贴现率下，1050 的现值是 1000。

所以这笔交易的期望净现值接近 0。

但这不代表它没有风险。

因为它仍然有 20% 概率只还 800。

所以我们得到一个非常重要的结论：

> 定价关心现值，风险关心分布。

## 22. 原语层结束检查

现在我们检查自己是否已经具备了搭建“组合逻辑”的基础。

类比计算机：

- 比特是最小信息单位。
- 逻辑门处理比特。
- 多个逻辑门组合成加法器。

在金融工程里：

- 时间点、现金流、价格、随机结果是原语。
- 贴现、利率、期望是基本操作。
- 多个现金流组合成债券、贷款、项目估值。
- 多个价格状态组合成衍生品 payoff。

所以，原语层结束时，我们应该能做到：

1. 表示时间。
2. 表示某个时间点的现金流。
3. 表示一组现金流。
4. 计算利息和利率。
5. 把未来金额贴现到现在。
6. 计算一组现金流的净现值。
7. 表示一个标的物的价格。
8. 表示未来的不确定结果。
9. 计算期望。
10. 手搓简单随机模拟。
11. 理解普通复利和连续复利。

如果这些都能做到，我们就可以进入组合层。

## 23. 进入下一层前的提醒：衍生品什么时候讲？

你可能会问：

> 期权、期货、远期这些常见衍生品，是下一层讲，还是再下一层讲？

建议节奏如下：

### 下一层：组合层

先讲最简单的远期合约。

因为远期合约可以用非常少的原语构造出来：

- 标的物现在价格
- 未来交割时间
- 约定交割价格
- 到期现金流
- 无套利思想

远期非常适合作为“第一个衍生品”。

### 组合层后半段

再讲期货和远期的区别。

但先不要急着进入保证金、逐日盯市、交易所制度。

先让学习者理解：

> 远期和期货都是关于未来买卖标的物的合约。

### 表示层或组合层末尾

再引入期权。

因为期权多了一个新抽象：

> Payoff 函数。

看涨期权到期收益可以写成：

$$\max(S_{T} - K, 0)$$

看跌期权到期收益可以写成：

$$\max(K - S_{T}, 0)$$

这里的 $S_{T}$ 是标的物到期价格，$K$ 是执行价。

期权最好不要在原语层讲太深。

原语层只需要埋下伏笔：

> 衍生品的价值，来自未来状态下的现金流。

## 24. 关于“复制任何衍生品”的一个小修正

你草稿里提到：

> 至少需要两种金融衍生品才可以复制任何衍生品。

更准确地说：

在最简单的一步二叉树模型里，如果未来只有两个状态：

- 上涨状态
- 下跌状态

那么通常只需要两个基础工具就可以复制任意一个依赖这两个状态的 payoff：

1. 标的资产
2. 无风险现金或无风险债券

它们不一定是两个衍生品。

更一般地，如果未来有 $n$ 个状态，那么要复制任意状态相关 payoff，通常需要 $n$ 个线性独立的基础资产。

所以后面的讲法可以是：

> 复制的本质不是“需要几个衍生品”，而是“需要足够多的独立基础工具来覆盖未来状态”。

这个思想可以留到二叉树定价时再正式展开。